<a href="https://colab.research.google.com/github/harsh3-web/Ecommerce-Recommender/blob/main/notebooks/03_improved_recommenders.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 · Improved recommenders

Fixes the weaknesses found in 02: **item-based CF** with cosine similarity on weighted interactions (browse = 1, purchase = 4), a **content model** using MiniLM sentence embeddings and a weighted user taste vector, and a **min-max normalised hybrid**.

> **Note:** saved outputs come from the first synthetic product catalogue (before Day 4). The 99% category match here is the inflated result that motivated the proper evaluation in 04.

In [ ]:

!git clone https://github.com/harsh3-web/Ecommerce-Recommender.git
%cd Ecommerce-Recommender
!pip install -q sentence-transformers

Cloning into 'Ecommerce-Recommender'...
remote: Enumerating objects: 17, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 17 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (17/17), 19.64 KiB | 1.96 MiB/s, done.
Resolving deltas: 100% (4/4), done.
/content/Ecommerce-Recommender


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!cp /content/drive/MyDrive/Ecommerce-Recommender-data/*.csv .
!ls *.csv

Mounted at /content/drive
browsing_history_expanded.csv  products_expanded.csv
ground_truth_tastes.csv        purchases_expanded.csv
product_images_expanded.csv    users_expanded.csv


In [ ]:
import numpy as np
import pandas as pd

users     = pd.read_csv("users_expanded.csv")
products  = pd.read_csv("products_expanded.csv")
purchases = pd.read_csv("purchases_expanded.csv")
browsing  = pd.read_csv("browsing_history_expanded.csv")
tastes    = pd.read_csv("ground_truth_tastes.csv").set_index("user_id")

In [ ]:
#  purchases show taste more strongly than browsing -> give them more weight
BROWSE_W, BUY_W = 1.0, 4.0

b = browsing[["user_id", "product_id"]].assign(weight=BROWSE_W)    # every browse = 1
p = purchases[["user_id", "product_id"]].assign(weight=BUY_W)      # every purchase = 4
# Stack both, then add up weights per (user, product) pair -> browsed AND bought = 1 + 4 = 5
inter = pd.concat([b, p]).groupby(["user_id", "product_id"], as_index=False)["weight"].sum()
inter.head()

,user_id,product_id,weight
0,1,19,1.0
1,1,25,1.0
2,1,32,5.0
3,1,41,1.0
4,1,48,1.0


In [ ]:
from scipy.sparse import csr_matrix   # sparse matrix: stores ONLY the non-zero cells (saves memory)

n_users, n_products = len(users), len(products)
rows = inter["user_id"].values - 1        # user 1 -> row 0 (Python counts from 0)
cols = inter["product_id"].values - 1     # product 1 -> column 0
R = csr_matrix((inter["weight"].values, (rows, cols)), shape=(n_users, n_products))
print(R.shape, "| non-zero cells:", R.nnz)   # (500, 200); nnz = number of filled cells

(500, 200) | non-zero cells: 5877


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# R.T (transpose): each ITEM becomes a row, described by which users interacted with it
# cosine similarity compares every item with every other item -> 200 x 200 table
item_sim = cosine_similarity(R.T)
np.fill_diagonal(item_sim, 0)      # an item is 100% similar to itself - remove that, it's useless
print(item_sim.shape)

(200, 200)


In [ ]:
pid = 1                                                  # product 1 (a T-Shirt)
top5 = np.argsort(item_sim[pid - 1])[::-1][:5] + 1       # argsort = indices sorted low->high; [::-1] reverses
print(products.loc[products["product_id"] == pid, ["product_name", "category"]])
products.set_index("product_id").loc[top5, ["product_name", "category"]]

           product_name  category
0  Blue Graphic T-Shirt  T-Shirts


,product_name,category
product_id,,
49,White Graphic T-Shirt,T-Shirts
177,Red Basic T-Shirt,T-Shirts
105,Beige Oversized T-Shirt,T-Shirts
185,Black Graphic T-Shirt,T-Shirts
65,Maroon Basic T-Shirt,T-Shirts


In [ ]:
def seen_items(uid):
    """
    Returns all products that the user has already interacted with
    (browsed or purchased), so they are not recommended again.
    """

    user_history = inter.loc[
        inter["user_id"] == uid,
        "product_id"
    ]

    return set(user_history)


def to_recs(scores, uid, source):
    """
    Converts product recommendation scores into a ranked recommendation table.

    Each score represents how suitable a product is for the user.
    Products already seen by the user are removed before returning results.
    """

    recs = products.copy()

    # Add recommendation score and model source information
    recs["score"] = scores
    recs["source"] = source

    # Remove products that the user has already browsed or purchased
    already_seen = seen_items(uid)
    recs = recs[~recs["product_id"].isin(already_seen)]

    # Sort recommendations from highest score to lowest score
    return recs.sort_values(
        by="score",
        ascending=False
    )

In [ ]:
##Item-based CF recommender
def item_cf(uid):
    user_vec = R[uid - 1].toarray().ravel()   # this user's row -> 200 weights (0 for untouched items)
    # For every product: sum over my items of (my weight for that item x its similarity to the product)
    scores = user_vec @ item_sim              # @ = matrix multiplication
    return to_recs(scores, uid, "Item-CF")

item_cf(7)[["product_id", "product_name", "category", "score"]].head(5)

,product_id,product_name,category,score
141,142,Grey Chronograph Shoes,Shoes,3.491124
26,27,Blue Straight Jeans,Jeans,3.444827
138,139,Navy Skinny Jeans,Jeans,3.331264
178,179,Brown Straight Jeans,Jeans,2.668130
116,117,White Bomber Jacket,Jackets,2.654965


In [ ]:
##Turn product text into vectors

from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer("all-MiniLM-L6-v2")   # small pretrained model: text -> 384 numbers
# Combine category + name + description into one text per product
texts = (products["category"] + ". " + products["product_name"] + ". " + products["description"]).tolist()
item_emb = encoder.encode(texts, normalize_embeddings=True)   # normalize = each vector has length 1
print(item_emb.shape)                                         # (200, 384)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

(200, 384)


In [ ]:
##Sanity check: text-similar products
pid = 1
sims = item_emb @ item_emb[pid - 1]                  # similarity of product 1 to ALL products
top5 = np.argsort(sims)[::-1][1:6] + 1               # [1:6] skips position 0 (product 1 itself)
print(products.loc[products["product_id"] == pid, ["product_name", "category"]])
products.set_index("product_id").loc[top5, ["product_name", "category"]]

           product_name  category
0  Blue Graphic T-Shirt  T-Shirts


,product_name,category
product_id,,
137,Blue Basic T-Shirt,T-Shirts
113,Blue Polo T-Shirt,T-Shirts
89,Blue Striped T-Shirt,T-Shirts
161,Beige Graphic T-Shirt,T-Shirts
25,Grey Striped T-Shirt,T-Shirts


In [ ]:
##Content-based recommender with a user "taste vector"
def content_emb(uid):
    mine = inter[inter["user_id"] == uid]                 # this user's products and weights
    w = mine["weight"].values[:, None]                    # weights as a column, shape (n, 1)
    vecs = item_emb[mine["product_id"].values - 1]        # embeddings of those products, shape (n, 384)
    profile = (w * vecs).sum(axis=0) / w.sum()            # WEIGHTED average -> the user's taste vector
    profile = profile / np.linalg.norm(profile)           # length 1 again, so dot = cosine
    return to_recs(item_emb @ profile, uid, "Content-Emb")

content_emb(7)[["product_id", "product_name", "category", "score"]].head(5)

,product_id,product_name,category,score
76,77,Grey Puffer Jacket,Jackets,0.829633
28,29,Grey Puffer Jacket,Jackets,0.813861
140,141,Beige Leather Jacket,Jackets,0.804125
100,101,Brown Denim Jacket,Jackets,0.801878
12,13,Beige Denim Jacket,Jackets,0.798400


In [ ]:
##Min-max scaling
def minmax(s):
    """Rescale scores to 0..1 so different methods become comparable."""
    spread = s.max() - s.min()
    return (s - s.min()) / spread if spread > 0 else s * 0   # avoid dividing by zero

print(minmax(pd.Series([2, 4, 10])).tolist())   # -> [0.0, 0.25, 1.0]

[0.0, 0.25, 1.0]


In [ ]:
##Weighted hybrid
def hybrid_v2(uid, alpha=0.5):
    cf = minmax(item_cf(uid).set_index("product_id")["score"])      # CF scores, scaled to 0..1
    cb = minmax(content_emb(uid).set_index("product_id")["score"])  # CB scores, scaled to 0..1
    combined = alpha * cf + (1 - alpha) * cb       # alpha=1 -> pure CF; alpha=0 -> pure content
    recs = products.set_index("product_id").loc[combined.index].reset_index()
    recs["score"], recs["source"] = combined.values, f"Hybrid a={alpha}"
    return recs.sort_values("score", ascending=False)

In [ ]:
#Same taste-match check as before
def taste_match(recommend_fn, user_ids):
    """Average % of top-10 recommendations that fall in the user's hidden liked categories."""
    results = []
    for u in user_ids:
        top10 = recommend_fn(u).head(10)                 # all new methods already return sorted results
        results.append(top10["category"].isin(set(tastes.loc[u])).mean())
    return np.mean(results)

In [ ]:
#Compare old and new
sample_users = tastes.index[:100]          # SAME 100 users as Day 1 -> fair comparison
print("Day 1 (old):  Collaborative 63% | Content-based 53% | Hybrid 53%")
print(f"Item-CF      : {taste_match(item_cf, sample_users):.0%}")
print(f"Content-Emb  : {taste_match(content_emb, sample_users):.0%}")
print(f"Hybrid a=0.5 : {taste_match(hybrid_v2, sample_users):.0%}")

Day 1 (old):  Collaborative 63% | Content-based 53% | Hybrid 53%
Item-CF      : 82%
Content-Emb  : 99%
Hybrid a=0.5 : 98%


In [ ]:
###Find the best alpha

# Try different mixes; lambda fixes alpha 'a' and passes only the user id
for a in [0.0, 0.25, 0.5, 0.75, 1.0]:
    score = taste_match(lambda u: hybrid_v2(u, alpha=a), sample_users)
    print(f"alpha = {a:.2f} -> {score:.0%}")

alpha = 0.00 -> 99%
alpha = 0.25 -> 99%
alpha = 0.50 -> 98%
alpha = 0.75 -> 92%
alpha = 1.00 -> 82%


## Day 2 results (taste match, top-10, same 100 users)
| Method          | Day 1 | Day 2 |
|-----------------|-------|-------|
| Collaborative   | 63%   | 82% (item-CF, cosine, browse=1/buy=4) |
| Content-based   | 53%   | 99% (MiniLM embeddings, weighted taste vector) |
| Hybrid          | 53%   | 98% (min-max normalised, alpha=0.5) |

Alpha sweep: 0.0->99%, 0.25->99%, 0.5->98%, 0.75->92%, 1.0->82%

Takeaway: 99% is inflated - the metric checks category match and the product
text contains the category name. Taste is purely category-based in synthetic data.
Need held-out evaluation (Recall@K, NDCG) on exact items -> Day 3.